# IEEE 118 EMT: composite and direct PiLine comparison

This report compares the original conventional composite network, the optimized composite network, a conventional network using the then-new standalone `PiLine_NoSub`, and the existing all-SSN network. The plots show component pre-step and post-step time, whole-step timing, and the residual after component pre/post work.

> **Historical result:** `PiLine_NoSub` has since been removed and its implementation moved into `EMT::Ph3::PiLine` behind an optional constructor. The standalone implementation's timing and its claim that it removed the internal midpoint are superseded; it was not trace-equivalent on the IEEE300 validation case. For current, trace-checked comparisons use `IEEE300_EMT_Publishable_Performance_Benchmark.ipynb`. The integrated direct PiLine keeps the MNA midpoint unknown, so it reduces component/task overhead, not matrix dimension.

## Simulation setup

All profiled runs use the same IEEE118 EMT3Ph case, 1.0 s simulated duration, 0.5 ms timestep (2,000 steps), MNA with KLU, sequential task timing, and no data logger. The new `PiLine_NoSub` run changes only the line implementation: each line directly stamps its three-phase Pi model and updates state without R/L/C child components or an internal virtual node. Loads, shunts, transformers, and generators remain conventional. The all-SSN run continues to use SSN lines, loads/shunts, and transformers with conventional generators.

Each profile is a single run. Instrumentation is enabled and adds overhead; treat small differences as indicative rather than benchmark-grade.

A five-run follow-up now compares the original dense RHS stamping path with direct compact RHS accumulation, labelled **SSN - post optimization**. Both variants use the same IEEE118 EMT3Ph case, 1.0 s simulation, 0.5 ms timestep, KLU, sequential per-task profiling, and no data logger. The five-run comparison is summarized separately below; the earlier four-variant plots retain their original single-run measurements.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

relative = Path("examples/Notebooks/Performance/data/IEEE118_Composite_Optimization")
local = Path.cwd() / "data/IEEE118_Composite_Optimization"
if local.is_dir():
    data_dir = local
else:
    root = next(
        (p for p in [Path.cwd(), *Path.cwd().parents] if (p / relative).is_dir()), None
    )
    if root is None:
        raise FileNotFoundError(f"Could not find {relative}")
    data_dir = root / relative

tasks = pd.read_csv(data_dir / "task_times.csv")
steps = pd.read_csv(data_dir / "step_times.csv")
runs = pd.read_csv(data_dir / "profile_runs.csv")
solver = pd.read_csv(data_dir / "solver_times.csv")

case_names = {
    "conventional_before": "Conventional before",
    "conventional_optimized": "Conventional optimized",
    "conventional_nosub": "Conventional PiLine_NoSub",
    "ssn_network": "SSN network",
}
case_order = list(case_names.values())
colors = {
    "Conventional before": "#4C78A8",
    "Conventional optimized": "#72A0C1",
    "Conventional PiLine_NoSub": "#59A14F",
    "SSN network": "#F28E2B",
}
component_order = ["Lines", "Loads + shunts", "Transformers", "Generators"]
phase_names = {"MnaPreStep": "Pre-step", "MnaPostStep": "Post-step"}
plt.rcParams.update({"font.size": 10, "axes.titlesize": 12, "axes.labelsize": 10})

tasks["case_label"] = tasks["case"].map(case_names)
tasks["task_ms"] = tasks["average_ns"] / 1e6
groups = (
    tasks[tasks.sample_count > 0]
    .groupby(["case", "case_label", "component_group", "phase"], as_index=False)
    .agg(ms_per_step=("task_ms", "sum"))
)
runs["case_label"] = runs["case"].map(case_names)
runs["mean_step_ms"] = runs["mean_step_s"] * 1000
solver["case_label"] = solver["case"].map(case_names)

## 1. Component pre-step time

Each bar sums the average pre-step time of tasks in that component group for one simulated step. The `PiLine_NoSub` run keeps the conventional component set and changes only line implementation.

In [ ]:
def plot_component_phase(phase, title):
    data = groups[groups.phase == phase]
    pivot = data.pivot_table(
        index="component_group",
        columns="case_label",
        values="ms_per_step",
        fill_value=0,
    )
    pivot = pivot.reindex(component_order).fillna(0).reindex(columns=case_order)
    fig, ax = plt.subplots(figsize=(12, 4.7))
    pivot.plot(kind="bar", ax=ax, color=[colors[c] for c in case_order], width=0.78)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Sum of task averages (ms/step)")
    ax.tick_params(axis="x", rotation=12)
    ax.grid(axis="y", alpha=0.22)
    ax.set_axisbelow(True)
    ax.legend(title=None, ncol=4, loc="upper right", frameon=False)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", padding=2, fontsize=8)
    fig.tight_layout()
    plt.show()


plot_component_phase("MnaPreStep", "Component pre-step time by group")

## 2. Component post-step time

This includes the component state/current updates after the network solve. In particular, compare line post-step time: `PiLine_NoSub` updates the branch state in one line task instead of dispatching the composite line’s child component tasks.

In [ ]:
plot_component_phase("MnaPostStep", "Component post-step time by group")

## 3. Whole simulation-step time

This wall timer covers the complete sequential scheduler step, including component tasks, solver task, and loop overhead.

In [ ]:
whole = runs.groupby(["case", "case_label"], as_index=False).agg(
    mean_step_ms=("mean_step_ms", "mean")
)
whole = whole.set_index("case_label").loc[case_order].reset_index()
fig, ax = plt.subplots(figsize=(10.5, 4.4))
bars = ax.bar(
    whole.case_label,
    whole.mean_step_ms,
    color=[colors[c] for c in whole.case_label],
    width=0.64,
)
ax.bar_label(bars, fmt="%.3f ms", padding=3, fontsize=9)
ax.set_ylabel("Mean whole-step wall time (ms/step)")
ax.set_title("Whole simulation-step time (profiled)")
ax.grid(axis="y", alpha=0.22)
ax.set_axisbelow(True)
ax.tick_params(axis="x", rotation=8)
fig.tight_layout()
plt.show()

## 4. Step time after component pre-step and post-step

The remainder is whole-step wall time minus the sum of measured component pre-step and post-step task averages. It includes the MNA solve, scheduler/loop overhead, and instrumentation; it is the closest available estimate of the integration/solver portion, not a pure numerical kernel timer.

In [ ]:
prepost = (
    groups[groups.phase.isin(["MnaPreStep", "MnaPostStep"])]
    .groupby(["case", "case_label"], as_index=False)
    .agg(component_prepost_ms=("ms_per_step", "sum"))
)
split = whole.merge(prepost, on=["case", "case_label"], how="left")
split["remaining_ms"] = split.mean_step_ms - split.component_prepost_ms
display(
    split[["case_label", "component_prepost_ms", "remaining_ms", "mean_step_ms"]].round(
        4
    )
)
fig, ax = plt.subplots(figsize=(10.5, 4.4))
bars = ax.bar(
    split.case_label,
    split.remaining_ms,
    color=[colors[c] for c in split.case_label],
    width=0.64,
)
ax.bar_label(bars, fmt="%.3f ms", padding=3, fontsize=9)
ax.set_ylabel("Whole step minus component pre/post (ms/step)")
ax.set_title("Remaining solver / integration-loop time")
ax.grid(axis="y", alpha=0.22)
ax.set_axisbelow(True)
ax.tick_params(axis="x", rotation=8)
fig.tight_layout()
plt.show()

## 5. Summary and direct solver timings

The `Solver task` row is the full MNA solve task measured by the scheduler. Direct KLU solve and matrix-refactorization times are reported separately by solver instrumentation.

In [ ]:
phase_totals = (
    tasks[tasks.phase.isin(["MnaPreStep", "MnaPostStep", "Solve"])]
    .groupby(["case", "phase"], as_index=False)
    .agg(avg_ns=("average_ns", "sum"))
)
phase_totals["ms_per_step"] = phase_totals.avg_ns / 1e6
summary = (
    runs[["case", "case_label", "logged_calculation_s", "mean_step_ms"]]
    .merge(
        phase_totals[phase_totals.phase == "MnaPreStep"][
            ["case", "ms_per_step"]
        ].rename(columns={"ms_per_step": "pre_step_ms"}),
        on="case",
    )
    .merge(
        phase_totals[phase_totals.phase == "MnaPostStep"][
            ["case", "ms_per_step"]
        ].rename(columns={"ms_per_step": "post_step_ms"}),
        on="case",
    )
    .merge(
        phase_totals[phase_totals.phase == "Solve"][["case", "ms_per_step"]].rename(
            columns={"ms_per_step": "solver_task_ms"}
        ),
        on="case",
    )
    .merge(split[["case", "remaining_ms"]], on="case")
)
summary = summary[
    [
        "case_label",
        "logged_calculation_s",
        "pre_step_ms",
        "post_step_ms",
        "solver_task_ms",
        "remaining_ms",
        "mean_step_ms",
    ]
]
summary.columns = [
    "Case",
    "Calculation time (s)",
    "Pre-step (ms)",
    "Post-step (ms)",
    "Solver task (ms)",
    "After pre/post (ms)",
    "Whole step (ms)",
]
summary = summary.set_index("Case").reindex(case_order).reset_index()
display(summary.round(4))

solver_view = solver[
    ["case_label", "klu_avg_s", "klu_refactor_total_s", "solve_count"]
].copy()
solver_view["KLU solve (µs)"] = solver_view.klu_avg_s * 1e6
solver_view["Refactorization total (s)"] = solver_view.klu_refactor_total_s
solver_view = solver_view.rename(
    columns={"case_label": "Case", "solve_count": "Solves"}
)
solver_view = solver_view.set_index("Case").reindex(case_order).reset_index()
display(
    solver_view[
        ["Case", "Solves", "KLU solve (µs)", "Refactorization total (s)"]
    ].round(4)
)

component_detail = (
    groups[groups.phase.isin(["MnaPreStep", "MnaPostStep"])]
    .pivot_table(
        index=["component_group", "phase"],
        columns="case_label",
        values="ms_per_step",
        fill_value=0,
    )
    .reindex(columns=case_order)
)
component_detail.index = component_detail.index.set_names(["Component group", "Phase"])
display(component_detail.round(4))

## Interpretation

`PiLine_NoSub` lowers the measured line post-step work from about 1.013 ms/step for the optimized composite to 0.301 ms/step. Its line pre-step time is 0.272 ms/step versus 0.582 ms/step for the optimized composite. Whole-step time falls from 3.648 to 2.403 ms/step, and logged calculation time falls from 7.30 to 4.81 s in these single profiled runs. The NoSub run is still slower than the all-SSN profile at 1.94 ms/step, whose lines, loads/shunts, and transformers all use SSN implementations.

The direct line retains the same three-phase R/L/C/G Pi model, timestep, and power-flow initialization path. It avoids the composite line’s child-component dispatch and virtual node. These timing results support the expected post-step improvement; compare simulation outputs over a logged run before treating the implementation as numerically validated.

## 6. SSN direct RHS accumulation: repeated timing comparison

The dense RHS baseline writes each PiLine/load contribution into its full-size component vector, then the solver rereads those entries. In **SSN - post optimization**, pre-step computes history terms and the solver accumulates them directly into the global RHS. Five 1-second runs were recorded for each variant with identical settings. Means and sample standard deviations are shown so normal run-to-run variation is visible.

In [ ]:
repeat_runs = pd.read_csv(data_dir / "rhs_optimization_repeat_runs.csv")
task_repeat = pd.read_csv(data_dir / "rhs_optimization_task_summary.csv")
repeat_summary = repeat_runs.groupby("case", as_index=False).agg(
    repetitions=("repetition", "count"),
    calculation_mean_s=("logged_calculation_s", "mean"),
    calculation_std_s=("logged_calculation_s", "std"),
    mean_step_ms=("mean_step_ms", "mean"),
    step_std_ms=("mean_step_ms", "std"),
)
display(repeat_runs.round(4))
display(repeat_summary.round(4))

fig, ax = plt.subplots(figsize=(8.5, 4.2))
bars = ax.bar(
    repeat_summary["case"],
    repeat_summary.calculation_mean_s,
    yerr=repeat_summary.calculation_std_s,
    capsize=5,
    color=["#F28E2B", "#E15759"],
    width=0.58,
)
ax.bar_label(bars, fmt="%.3f s", padding=4, fontsize=9)
ax.set_ylabel("Calculation time for 1 simulated second (s)")
ax.set_title("Five-run calculation-time comparison (mean ± SD)")
ax.grid(axis="y", alpha=0.22)
ax.set_axisbelow(True)
ax.tick_params(axis="x", rotation=8)
fig.tight_layout()
plt.show()

In [ ]:
task_order = [
    "Lines · Pre-step",
    "Lines · Post-step",
    "Loads + shunts · Pre-step",
    "Loads + shunts · Post-step",
    "Solver task · Solve",
]
task_repeat["label"] = task_repeat.component_group + " · " + task_repeat.phase
task_pivot = task_repeat.pivot(index="label", columns="case", values="mean_ms_per_step")
task_pivot = task_pivot.reindex(task_order)
task_pivot.plot(
    kind="bar",
    figsize=(10, 4.6),
    color=["#F28E2B", "#E15759"],
    width=0.74,
    yerr=task_repeat.pivot(
        index="label", columns="case", values="std_ms_per_step"
    ).reindex(task_order),
    capsize=3,
)
plt.ylabel("Sum of task averages (ms/step; mean ± SD)")
plt.title("Line/load pre-step and post-step, plus solver task")
plt.xticks(rotation=12, ha="right")
plt.grid(axis="y", alpha=0.22)
plt.gca().set_axisbelow(True)
plt.legend(title=None, frameon=False)
plt.tight_layout()
plt.show()

**Result:** the dense baseline averaged 3.799 s and the post-optimization run averaged 3.884 s of calculation time (+2.2% in this batch). The run-to-run standard deviations were 0.184 s and 0.188 s, respectively, so this difference is within the observed timing spread; the measurement does not demonstrate a speedup. The whole-step mean moved from 1.898 to 1.941 ms. Component task timing shows a small reduction in line/load pre-step work, but no consistent gain in total step time. The direct accumulation change should therefore be kept only if a later unprofiled or interleaved benchmark confirms a benefit.